In [13]:
from dotenv import load_dotenv
from pathlib import Path
import os
from google import genai
import pandas as pd

# Load env
load_dotenv(dotenv_path=Path("../.env"))

# New client setup
client = genai.Client(api_key=os.environ.get("GEMINI_API_KEY"))

df = pd.read_csv("../data/processed/oil_production_features.csv")

def format_data_for_prompt(df, country=None):
    """Format oil production data as text for the prompt."""
    if country:
        data = df[df['country'] == country.upper()].copy()
    else:
        data = df.copy()
    
    summary = f"""
Oil Production Dataset Summary:
- Countries included: {df['country'].nunique()} countries
- Year range: {df['year'].min()} to {df['year'].max()}
- Total records: {len(df)}
"""
    if country:
        summary += f"""
Country: {country.upper()}
- Years available: {data['year'].min()} to {data['year'].max()}
- Average production: {data['production'].mean():,.1f} thousand barrels/day
- Peak production: {data['production'].max():,.1f} (in {data.loc[data['production'].idxmax(), 'year']})
- Lowest production: {data['production'].min():,.1f} (in {data.loc[data['production'].idxmin(), 'year']})
- Latest 3-year rolling avg: {data['rolling_3'].iloc[-1]:,.1f}

Year-by-year production:
{data[['year', 'production', 'rolling_3']].to_string(index=False)}
"""
    else:
        top5 = df.groupby('country')['production'].mean().nlargest(5)
        summary += f"""
Top 5 Countries by Average Production:
{top5.to_string()}

Overall Stats:
- Highest single production value: {df['production'].max():,.1f} ({df.loc[df['production'].idxmax(), 'country']}, {df.loc[df['production'].idxmax(), 'year']})
- Global average production: {df['production'].mean():,.1f}
"""
    return summary


def build_prompt(data_summary, user_question):
    return f"""You are an oil production data analyst.

You have been given the following oil production data:

{data_summary}

Answer the following question using ONLY the data provided above.
If the answer cannot be determined from the data, say so clearly.
Be concise and specific — include numbers where relevant.
Do not speculate beyond what the data shows.

Question: {user_question}

Answer:"""


def ask_question(data_summary, user_question):
    prompt = build_prompt(data_summary, user_question)
    response = client.models.generate_content(
        model="gemini-2.5-flash",  
        contents=prompt
    )
    
    return response.text


In [14]:
# Question 1
summary = format_data_for_prompt(df, country="AGO")
answer = ask_question(summary, "Which year had the highest production for Angola?")
print(answer)

The highest production for Angola was in 2008.


In [15]:
# Guardrail Test
answer = ask_question(summary, "Why did Angola's production drop in 1990?")
print(answer)

Based on the provided data, Angola's production did not drop in 1990; it increased from 22,765.116 thousand barrels/day in 1989 to 23,826.950 thousand barrels/day in 1990. Therefore, the reason for a drop cannot be determined as no drop occurred according to the data.


In [16]:
# Comparison question
summary = format_data_for_prompt(df)
answer = ask_question(summary, "Which country has the highest average oil production?")
print(answer)

Russia (RUS) has the highest average oil production, with an average of 415,368.97.


In [17]:
# Out-of-scope test (model should refuse to speculate)
answer = ask_question(summary, "What will oil prices be next year?")
print(answer)

The provided data contains information on oil production volume and related statistics, but it does not include any information about oil prices, historical or future. Therefore, the question of what oil prices will be next year cannot be determined from the data provided.


In [ ]:
def ask_question_interactive(df):
    print("🛢️ Oil Production Data Analyst")
    print("=" * 40)
    
    # Ask for country filter
    country = input("Enter a country code to filter (e.g. AGO, SAU) or press Enter for all countries: ").strip()
    
    # Generate summary
    if country:
        summary = format_data_for_prompt(df, country=country)
    else:
        summary = format_data_for_prompt(df)
    
    print("\nData loaded! You can now ask questions.")
    print("Type 'quit' to exit.\n")
    
    # Q&A loop
    while True:
        question = input("Your question: ").strip()
        
        if question.lower() == 'quit':
            print("Goodbye!")
            break
        
        if not question:
            print("Please enter a question.\n")
            continue
        
        print("\nThinking...\n")
        answer = ask_question(summary, question)
        print(f"Answer: {answer}")
        print("-" * 40 + "\n")

ask_question_interactive(df)

🛢️ Oil Production Data Analyst


KeyboardInterrupt: Interrupted by user